<a href="https://colab.research.google.com/github/Khajavi8056/Hip/blob/main/Hipotensertrade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:

# نصب کتابخانه‌های اصلی
!pip install streamlit tensortrade yfinance pandas numpy matplotlib scikit-learn stable-baselines3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.6/32.6 MB 57.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 958.1/958.1 kB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 103.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 115.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   

App.py

In [6]:

%%writefile app.py
import streamlit as st
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# TensorTrade Imports (نسخه 0.3.0)
from tensortrade.env import TradingEnvironment
from tensortrade.data import DataFeed, Stream
from tensortrade.strategies import StableBaselinesTradingStrategy
from tensortrade.agents import PPOAgent, A2CAgent, DQNAgent
from tensortrade.instruments import USD, BTC
from tensortrade.wallets import Wallet, Portfolio

# تنظیمات Streamlit
st.set_page_config(page_title="TensorTrade Pro", layout="wide")
st.title("TensorTrade Advanced Trading System")

# ======================== تنظیمات سایدبار ========================
with st.sidebar:
    st.header("📊 تنظیمات داده")
    data_source = st.radio("منبع داده:", ["Yahoo Finance", "آپلود فایل CSV"])

    if data_source == "Yahoo Finance":
        ticker = st.text_input("نماد (مثال: BTC-USD)", "BTC-USD")
        start_date = st.date_input("تاریخ شروع", pd.to_datetime("2020-01-01"))
        end_date = st.date_input("تاریخ پایان", pd.to_datetime("2023-01-01"))
    else:
        uploaded_file = st.file_uploader("فایل CSV را آپلود کنید", type="csv")
        date_col = st.text_input("ستون تاریخ (Date)", "Date")
        price_col = st.text_input("ستون قیمت پایانی (Close)", "Close")

    st.header("⚙️ تقسیم داده")
    train_size = st.slider("درصد داده آموزش", 50, 90, 70)
    val_size = st.slider("درصد داده اعتبارسنجی", 5, 30, 15)
    test_size = 100 - train_size - val_size
    st.write(f"داده تست: {test_size}%")

    st.header("🧠 پارامترهای مدل")
    model_name = st.selectbox("الگوریتم", ["PPO", "A2C", "DQN"])

    # تنظیمات پیشرفته شبکه عصبی
    st.subheader("ساختار شبکه عصبی")
    num_layers = st.slider("تعداد لایه‌های پنهان", 1, 5, 2)
    hidden_units = st.text_input("تعداد نورون‌ها در هر لایه (جدا با کاما)", "64, 32")
    activation = st.selectbox("تابع فعالسازی", ["relu", "tanh", "sigmoid"])
    learning_rate = st.number_input("نرخ یادگیری", 0.0001, 0.1, 0.001, step=0.0001)

    st.header("📉 پارامترهای بکتست")
    initial_balance = st.number_input("موجودی اولیه (USD)", 1000, 1000000, 10000)

# ======================== پردازش داده ========================
@st.cache_data
def load_data():
    if data_source == "Yahoo Finance":
        data = yf.download(ticker, start=start_date, end=end_date)
        data = data[['Open', 'High', 'Low', 'Close', 'Volume']]
    else:
        data = pd.read_csv(uploaded_file)
        data[date_col] = pd.to_datetime(data[date_col])
        data.set_index(date_col, inplace=True)
        data = data[[price_col]]
        data.columns = ['Close']

    # تقسیم داده به Train/Val/Test
    train_data, temp_data = train_test_split(data, train_size=train_size/100, shuffle=False)
    val_data, test_data = train_test_split(temp_data, test_size=test_size/(test_size+val_size), shuffle=False)
    return train_data, val_data, test_data

try:
    train_data, val_data, test_data = load_data()
    st.success("✅ داده‌ها با موفقیت بارگذاری شدند!")

    # نمایش داده‌ها
    st.subheader("📈 نمودار قیمت")
    fig, ax = plt.subplots()
    ax.plot(train_data['Close'], label='آموزش')
    ax.plot(val_data['Close'], label='اعتبارسنجی')
    ax.plot(test_data['Close'], label='تست')
    ax.legend()
    st.pyplot(fig)

    # ======================== ساخت محیط معاملاتی ========================
    def create_environment(data):
        price = Stream('close', data['Close'].values)
        feed = DataFeed([price])

        portfolio = Portfolio(
            USD,
            wallets=[
                Wallet(exchange=None, instrument=USD, balance=initial_balance),
                Wallet(exchange=None, instrument=BTC, balance=0)
            ]
        )

        return TradingEnvironment(
            portfolio=portfolio,
            feed=feed,
            window_size=20,
            action_scheme='discrete',
            reward_scheme='risk-adjusted'
        )

    env_train = create_environment(train_data)
    env_val = create_environment(val_data)
    env_test = create_environment(test_data)

    # ======================== آموزش مدل ========================
    if st.button("🚀 شروع آموزش و بکتست"):
        # تنظیمات شبکه عصبی
        net_arch = [int(x.strip()) for x in hidden_units.split(',')]

        # انتخاب کلاس مدل
        agent_class = {
            "PPO": PPOAgent,
            "A2C": A2CAgent,
            "DQN": DQNAgent
        }[model_name]

        # ساخت استراتژی
        strategy = StableBaselinesTradingStrategy(
            environment=env_train,
            agent_class=agent_class,
            agent_params={
                'policy': 'MlpPolicy',
                'policy_kwargs': {
                    'net_arch': net_arch,
                    'activation_fn': eval(f"torch.nn.{activation}")
                },
                'learning_rate': learning_rate
            }
        )

        # آموزش روی داده Train
        with st.spinner("در حال آموزش مدل..."):
            strategy.train(steps=5000)
            strategy.save("trained_model")

        # اعتبارسنجی روی داده Val
        with st.spinner("در حال اعتبارسنجی..."):
            val_performance = strategy.run(env_val)

        # تست نهایی روی داده Test
        with st.spinner("در حال اجرای بکتست نهایی..."):
            test_performance = strategy.run(env_test)

        # ======================== نمایش نتایج ========================
        st.subheader("📊 نتایج نهایی")

        col1, col2, col3 = st.columns(3)
        with col1:
            st.metric("سود/زیان (Train)", f"{env_train.portfolio.performance.profit_loss:.2f}%")
        with col2:
            st.metric("سود/زیان (Val)", f"{val_performance['profit_loss']:.2f}%")
        with col3:
            st.metric("سود/زیان (Test)", f"{test_performance['profit_loss']:.2f}%")

        # نمودار ارزش پرتفوی
        st.subheader("📈 نمودار ارزش پرتفوی")
        fig, ax = plt.subplots()
        ax.plot(env_train.portfolio.performance.net_worth, label='آموزش')
        ax.plot(val_performance['net_worth'], label='اعتبارسنجی')
        ax.plot(test_performance['net_worth'], label='تست')
        ax.legend()
        st.pyplot(fig)

        # ذخیره‌سازی کامل مدل
        st.subheader("💾 ذخیره مدل")
        model_bytes = open("trained_model.zip", "rb").read()
        st.download_button(
            label="دانلود مدل آموزش‌دیده",
            data=model_bytes,
            file_name="trading_model.zip",
            mime="application/zip"
        )

        # ذخیره تنظیمات
        config = {
            "hidden_layers": net_arch,
            "activation": activation,
            "learning_rate": learning_rate
        }
        st.download_button(
            label="دانلود تنظیمات مدل",
            data=json.dumps(config),
            file_name="model_config.json",
            mime="application/json"
        )

except Exception as e:
    st.error(f"❌ خطا: {str(e)}")

Overwriting app.py


اجرا در Google Colab

In [11]:
!streamlit run app.py --server.port 8501




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.142.187.115:8501

  Stopping...
  Stopping...


In [15]:
!npm install -g localtunnel
!lt --port :8501

# ... (Your Streamlit code here) ...

!streamlit run app.py --server.port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
added 22 packages in 3s
⠋
⠋3 packages are looking for funding
⠋  run `npm fund` for details
⠋Usage: lt --port [num] <options>

Options:
  -p, --port                Internal HTTP server port                 [required]
  -h, --host                Upstream server providing forwarding
                                             [default: "https://localtunnel.me"]
  -s, --subdomain           Request this subdomain
  -l, --local-host          Tunnel traffic to this host instead of localhost,
                            override Host header to this host
      --local-https         Tunnel traffic to a local HTTPS server     [boolean]
      --local-cert          Path to certificate PEM file for local HTTPS server
      --local-key           Path to certificate key file for local HTTPS server
      --local-ca            Path to certificate authority file for self-signed
                            certificates
      --allow-invalid-cert  Disable certificate checks

In [16]:
from IPython import display

# Display the Streamlit app in an iframe
display.IFrame(src=public_url, width="100%", height=1000)

NameError: name 'public_url' is not defined

In [14]:
!sudo lsof -i :8501